# Foundation models for tabular data

My whole academic experience was focused on tabular data in one way or another, whereas I have spent most of my career in industry working with computer vision foundation models. The recent implementations of **tabular foundation models** (TFMs) therefore sparked quite some curiosity and a little astonishment on my end.

This notebook is simply a brief overview of current TFMs that are based on **prior-data fitted networks** (PFNs) [[Müller et al., 2022](https://arxiv.org/abs/2112.10510); [Hollmann et al., 2022](https://arxiv.org/abs/2207.01848)]. After such a model has been pre-trained on large amounts of synthetic data, it can be used on real data by utilizing a form of **in-context learning** (ICL).

## Prior-data fitted networks

PFNs have emerged in the context of Bayesian inference for a supervised ML problem [[Müller et al., 2022](https://arxiv.org/abs/2112.10510); [Nagler, 2023](https://arxiv.org/abs/2305.11097)]. Rather than explicitly computing the posterior distribution of the unknown parameters, a transformer is used to approximate the corresponding predictive distribution directly.

To this end, one trains an appropriate PFN model over a prior distribution of synthetic tabular datasets. This prior embodies certain structural assumptions about data-generating mechanisms. It needs to be specified in a way that is amenable to sampling. A PFN can then be pre-trained with a large number of synthetically generated datasets from the prior.

After pre-training has been completed on synthetic data, one can compute predictions for unseen real datasets by passing both the train and test split into the PFN.

## Posterior predictive distribution

As a brief reminder, consider a standard supervised learning task where one wants to find a model predicting certain target variables $\boldsymbol{y}$ for provided values of the inputs $\boldsymbol{x}$. Given an iid dataset $D_{\mathrm{train}} = \{(\boldsymbol{x}_i, \boldsymbol{y}_i)\}_{i=1}^N$ of realized inputs and targets, one can infer the unknown parameters $\boldsymbol{\phi}$ of a statistical data model $p(D_{\mathrm{train}} | \boldsymbol{\phi})$.

Adapting a Bayesian inference approach to this task, one starts by eliciting a **prior distribution** $p(\boldsymbol{\phi})$ on the unknown parameters. This yields the joint distribution $p(D_{\mathrm{train}}, \boldsymbol{\phi}) = p(D_{\mathrm{train}} | \boldsymbol{\phi}) \, p(\boldsymbol{\phi})$. The **posterior distribution** $p(\boldsymbol{\phi} | D_{\mathrm{train}})$ then summarizes the information about the model parameters after conditioning on the observed data. It is obtained via Bayes' theorem
$$
p(\boldsymbol{\phi} | D_{\mathrm{train}}) =
\frac{p(D_{\mathrm{train}} | \boldsymbol{\phi}) \, p(\boldsymbol{\phi})}{p(D_{\mathrm{train}})}, \quad
p(D_{\mathrm{train}}) = \int p(D_{\mathrm{train}} | \boldsymbol{\phi}) \, p(\boldsymbol{\phi}) \, d \boldsymbol{\phi}.
$$

Following this, one usually wants to make predictions for a new input $\boldsymbol{x}_{\mathrm{test}}$. In the Bayesian framework, this is usually accomplished by computing the **posterior predictive distribution** (PPD)
$$
p(\boldsymbol{y}_{\mathrm{test}} | \boldsymbol{x}_{\mathrm{test}}, D_{\mathrm{train}}) =
\int p(\boldsymbol{y}_{\mathrm{test}} | \boldsymbol{x}_{\mathrm{test}}, \boldsymbol{\phi}) \,
p(\boldsymbol{\phi} | D_{\mathrm{train}}) \, d \boldsymbol{\phi} \propto
\int p(\boldsymbol{y}_{\mathrm{test}} | \boldsymbol{x}_{\mathrm{test}}, \boldsymbol{\phi}) \,
p(D_{\mathrm{train}} | \boldsymbol{\phi}) \, p(\boldsymbol{\phi}) \, d \boldsymbol{\phi}.
$$
This probability distribution represents the uncertainty of the predictions $\boldsymbol{y}_{\mathrm{test}}$ in fully Bayesian fashion.

## Prior fitting

Now we shift the focus away from the explicit posterior for a specific train set to the **prior distribution of tabular datasets**. In the probabilistic setup outlined above, this prior $p(D)$ over plausible datasets $D$ has the form
$$
p(D) = \int p(D | \boldsymbol{\phi}) \, p(\boldsymbol{\phi}) \, d \boldsymbol{\phi}.
$$
This notation also covers the case where one splits the dataset $D = D_{\mathrm{train}} \cup D_{\mathrm{test}}$ into a train and test part. For simplicity, below we consider a single test sample $D_{\mathrm{test}} = \{(\boldsymbol{x}_{\mathrm{test}}, \boldsymbol{y}_{\mathrm{test}})\}$ only.

The main idea of PFNs is to find the parameters $\boldsymbol{\theta}$ of a neural network $q_{\boldsymbol{\theta}}(\boldsymbol{y}_{\mathrm{test}} | \boldsymbol{x}_{\mathrm{test}}, D_{\mathrm{train}})$ such that it best approximates the PPD over the dataset prior. This is referred to as **prior fitting** and can be accomplished by minimizing the **prior-data negative log-likelihood**
$$
\ell(\boldsymbol{\theta}) =
\mathbb{E}_{D_{\mathrm{train}}, \{(\boldsymbol{x}_{\mathrm{test}}, \boldsymbol{y}_{\mathrm{test}})\}}
\left[ - \log q_{\boldsymbol{\theta}}(\boldsymbol{y}_{\mathrm{test}} | \boldsymbol{x}_{\mathrm{test}}, D_{\mathrm{train}}) \right].
$$

As usual, one can show that minimizing such a negative log-likelihood is equivalent to minimizing the **expected KL divergence** of $p(\cdot | \boldsymbol{x}_{\mathrm{test}}, D_{\mathrm{train}})$ from $q_{\boldsymbol{\theta}}(\cdot | \boldsymbol{x}_{\mathrm{test}}, D_{\mathrm{train}})$:
$$
\begin{align*}
\hat{\boldsymbol{\theta}} &= \operatorname*{argmin}_{\boldsymbol{\theta}}
\mathbb{E}_{D_{\mathrm{train}}, \boldsymbol{x}_{\mathrm{test}}} \left[
\operatorname{KL}(p(\cdot | \boldsymbol{x}_{\mathrm{test}}, D_{\mathrm{train}}),
q_{\boldsymbol{\theta}}(\cdot | \boldsymbol{x}_{\mathrm{test}}, D_{\mathrm{train}})) \right] \\ &=
\operatorname*{argmin}_{\boldsymbol{\theta}}
\mathbb{E}_{D_{\mathrm{train}}, \boldsymbol{x}_{\mathrm{test}}} \left[
\operatorname{H}(p(\cdot | \boldsymbol{x}_{\mathrm{test}}, D_{\mathrm{train}}),
q_{\boldsymbol{\theta}}(\cdot | \boldsymbol{x}_{\mathrm{test}}, D_{\mathrm{train}})) \right] \\ &=
\operatorname*{argmin}_{\boldsymbol{\theta}}
\mathbb{E}_{D_{\mathrm{train}}, \boldsymbol{x}_{\mathrm{test}}, \boldsymbol{y}_{\mathrm{test}}}
\left[ - \log q_{\boldsymbol{\theta}}(\boldsymbol{y}_{\mathrm{test}} | \boldsymbol{x}_{\mathrm{test}}, D_{\mathrm{train}}) \right].
\end{align*}
$$
This highlights how the PFN approximates the PPD over the prior distribution of datasets.

Of course, one can generalize the derivation above for a single test sample to multiple test samples $D_{\mathrm{test}} = \{(\boldsymbol{x}_j, \boldsymbol{y}_j)\}_{j=1}^M$. The corresponding objective then reads
$$
\ell(\boldsymbol{\theta}) = \mathbb{E}_{D_{\mathrm{train}}, D_{\mathrm{test}}}
\left[ - \sum_{j=1}^M \log q_{\boldsymbol{\theta}}(\boldsymbol{y}_j | \boldsymbol{x}_j, D_{\mathrm{train}}) \right].
$$